# Whole-Word Stress Exploration: **pronunciation** (User vs Merriam-Webster Baseline)

This notebook implements a deterministic DSP stress evaluation workflow for one word (Mode 2: whole-word).

It compares:
- **User recording** (`data/pronunciation/pronunciation_full.wav`)
- **Merriam-Webster baseline** (`data/pronunciation/pronunciation_mw_16k.wav`)
- **Expected stress pattern** from CMUdict-style phones (`AH0 AH2 IY0 EY1 AH0`)

Design constraints followed in this notebook:
- Relative prominence (not absolute amplitude)
- Baseline loudness normalized to user RMS
- Envelope alignment before comparison
- Signal quality gate to avoid harsh feedback on low-quality audio
- No emotion inference, no ML inference


In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display


In [ ]:
# Configuration
SR = 16000
FRAME_MS = 25
HOP_MS = 10
FRAME_LEN = int(SR * FRAME_MS / 1000)
HOP_LEN = int(SR * HOP_MS / 1000)

WORD = "pronunciation"
USER_PATH = Path("../data/pronunciation/pronunciation_full.wav")
BASELINE_PATH = Path("../data/pronunciation/pronunciation_mw_16k.wav")
VOWEL_META_PATH = Path("pronunciation_mw_16k_vowels.json")


In [ ]:
def load_mono(path: Path, sr=16000):
    x, _ = librosa.load(path.as_posix(), sr=sr, mono=True)
    return x.astype(np.float32)


def trim_silence(x, top_db=30):
    y, idx = librosa.effects.trim(x, top_db=top_db)
    return y.astype(np.float32), idx


def global_rms(x):
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))


def normalize_to_user_rms(user_x, base_x):
    scale = global_rms(user_x) / (global_rms(base_x) + 1e-12)
    return (base_x * scale).astype(np.float32), float(scale)


def rms_envelope(x, frame_len=FRAME_LEN, hop_len=HOP_LEN):
    env = librosa.feature.rms(y=x, frame_length=frame_len, hop_length=hop_len, center=True)[0]
    return env.astype(np.float32)


def robust_env_norm(env):
    p95 = np.percentile(env, 95)
    return env / (p95 + 1e-9)


def estimate_snr_db(raw_x, trimmed_idx, sr=16000, noise_ms=150):
    n = len(raw_x)
    edge = int(sr * noise_ms / 1000)
    left = raw_x[:min(edge, n)]
    right = raw_x[max(0, n-edge):]
    noise = np.concatenate([left, right]) if len(left) and len(right) else raw_x
    noise_rms = global_rms(noise)

    s0, s1 = trimmed_idx
    speech = raw_x[s0:s1] if s1 > s0 else raw_x
    speech_rms = global_rms(speech)
    snr_db = 20 * np.log10((speech_rms + 1e-12) / (noise_rms + 1e-12))
    return float(snr_db), float(speech_rms), float(noise_rms)


In [ ]:
# Load and pre-process audio
u_raw = load_mono(USER_PATH, SR)
mw_raw = load_mono(BASELINE_PATH, SR)

u_trim, u_idx = trim_silence(u_raw, top_db=30)
mw_trim, mw_idx = trim_silence(mw_raw, top_db=30)

mw_norm, scale = normalize_to_user_rms(u_trim, mw_trim)

print(f"User samples (raw/trim): {len(u_raw)} / {len(u_trim)}")
print(f"Baseline samples (raw/trim): {len(mw_raw)} / {len(mw_trim)}")
print(f"Baseline RMS scale -> user RMS: {scale:.3f}")


In [ ]:
# Build envelopes and normalize to relative scale
u_env = rms_envelope(u_trim)
mw_env = rms_envelope(mw_norm)

u_env_n = robust_env_norm(u_env)
mw_env_n = robust_env_norm(mw_env)

print(f"Envelope frames (user/base): {len(u_env_n)} / {len(mw_env_n)}")


In [ ]:
# DTW alignment between baseline and user envelopes (deterministic)
D, wp = librosa.sequence.dtw(X=mw_env_n[None, :], Y=u_env_n[None, :], metric="euclidean")
wp = wp[::-1]  # forward in time, each row is [mw_frame, user_frame]

# Baseline frame -> user frame mapping via median assignment
mapping = {}
for m_i, u_i in wp:
    mapping.setdefault(int(m_i), []).append(int(u_i))

mw_to_user = np.full(len(mw_env_n), np.nan, dtype=np.float32)
for i in range(len(mw_env_n)):
    if i in mapping:
        mw_to_user[i] = np.median(mapping[i])

# Fill unmapped frames by interpolation
valid = np.isfinite(mw_to_user)
mw_to_user = np.interp(np.arange(len(mw_to_user)), np.flatnonzero(valid), mw_to_user[valid])

print(f"DTW total cost: {float(D[-1,-1]):.3f}")


In [ ]:
# Visual checks: waveform and envelopes
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=False)

librosa.display.waveshow(u_trim, sr=SR, ax=ax[0], alpha=0.8, label="User")
librosa.display.waveshow(mw_norm, sr=SR, ax=ax[0], color="tab:orange", alpha=0.7, label="MW normalized")
ax[0].set_title("Trimmed waveforms (baseline RMS-normalized to user)")
ax[0].legend()

ax[1].plot(u_env_n, label="User envelope (norm)", linewidth=2)
ax[1].plot(mw_env_n, label="MW envelope (norm)", linewidth=2, alpha=0.8)
ax[1].set_title("RMS envelopes before DTW warping")
ax[1].set_xlabel("Frame")
ax[1].set_ylabel("Relative loudness")
ax[1].legend()
plt.tight_layout()


In [ ]:
# Read vowel anchors (CMUdict-like phones with stress digits)
vowel_meta = json.loads(VOWEL_META_PATH.read_text())
vowels = vowel_meta["vowels"]

for i, v in enumerate(vowels):
    print(i, v)

expected_primary_idx = next(i for i, v in enumerate(vowels) if v["stress"] == 1)
print("Expected primary stress syllable index:", expected_primary_idx)


In [ ]:
def sec_to_frame(t, sr=SR, hop_len=HOP_LEN, n_frames=None):
    idx = int(round((t * sr) / hop_len))
    if n_frames is None:
        return idx
    return int(np.clip(idx, 0, n_frames - 1))


def interval_prominence(env, start_f, end_f):
    if end_f <= start_f:
        end_f = start_f + 1
    seg = env[start_f:end_f]
    if len(seg) == 0:
        return 0.0
    # robust prominence estimate (captures nucleus energy while resisting spikes)
    return float(np.percentile(seg, 90))


In [ ]:
# Compute user/baseline prominence per vowel by mapping baseline intervals through DTW
records = []
for i, v in enumerate(vowels):
    m0 = sec_to_frame(v["t0"], n_frames=len(mw_env_n))
    m1 = sec_to_frame(v["t1"], n_frames=len(mw_env_n))
    if m1 <= m0:
        m1 = min(m0 + 1, len(mw_env_n) - 1)

    # mapped user interval (from baseline frame interval -> user frame interval)
    u_frames = mw_to_user[m0:m1]
    u0 = int(np.floor(np.min(u_frames)))
    u1 = int(np.ceil(np.max(u_frames)))
    u0 = int(np.clip(u0, 0, len(u_env_n)-1))
    u1 = int(np.clip(max(u1, u0+1), 1, len(u_env_n)))

    base_prom = interval_prominence(mw_env_n, m0, m1)
    user_prom = interval_prominence(u_env_n, u0, u1)

    records.append({
        "idx": i,
        "phone": v["phone"],
        "stress": int(v["stress"]),
        "baseline_frame_range": [m0, m1],
        "user_frame_range": [u0, u1],
        "baseline_prom": base_prom,
        "user_prom": user_prom,
    })

records


In [ ]:
# Determine observed primary stress from user prominence profile
user_prom = np.array([r["user_prom"] for r in records], dtype=np.float32)
base_prom = np.array([r["baseline_prom"] for r in records], dtype=np.float32)

observed_primary_idx = int(np.argmax(user_prom))
match = observed_primary_idx == expected_primary_idx

# Confidence: dominance of the selected syllable + match boost/penalty
sorted_user = np.sort(user_prom)[::-1]
top = float(sorted_user[0])
second = float(sorted_user[1]) if len(sorted_user) > 1 else 0.0
dominance = (top - second) / (top + 1e-9)

if match:
    confidence = 0.55 + 0.45 * np.clip(dominance * 2.0, 0.0, 1.0)
else:
    confidence = 0.15 + 0.35 * (1.0 - np.clip(dominance * 2.0, 0.0, 1.0))

confidence = float(np.clip(confidence, 0.0, 1.0))

snr_db, speech_rms, noise_rms = estimate_snr_db(u_raw, u_idx, sr=SR)
low_quality = (snr_db < 10.0) or (speech_rms < 0.01)

diagnostic = {
    "word": WORD,
    "mode": "whole_word_mode_2",
    "expected_primary_stress_syllable": expected_primary_idx,
    "observed_primary_stress_syllable": observed_primary_idx,
    "stress_match": bool(match),
    "confidence": round(confidence, 3),
    "signal_quality": {
        "snr_db_est": round(snr_db, 2),
        "speech_rms": round(speech_rms, 5),
        "noise_rms": round(noise_rms, 5),
        "low_quality": bool(low_quality),
    },
    "relative_prominence": [
        {
            "idx": r["idx"],
            "phone": r["phone"],
            "expected_stress": r["stress"],
            "baseline_prom": round(r["baseline_prom"], 4),
            "user_prom": round(r["user_prom"], 4),
        }
        for r in records
    ],
    "policy": {
        "no_emotion_inference": True,
        "no_ml_inference": True,
        "relative_loudness_based": True,
        "baseline_rms_normalized_to_user": True,
        "envelope_alignment_applied": True,
    },
}

diagnostic


In [ ]:
# Plot vowel intervals + prominence for interpretability
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(u_env_n, label="User env", linewidth=2)
ax.plot(mw_env_n, label="MW env", linewidth=1.8, alpha=0.75)

for r in records:
    u0, u1 = r["user_frame_range"]
    ax.axvspan(u0, u1, alpha=0.12, color="tab:green" if r["idx"] == expected_primary_idx else "tab:blue")
    ax.text((u0+u1)/2, max(u_env_n)*0.92, f"{r['idx']}:{r['phone']}", ha="center", fontsize=8)

ax.set_title("User envelope with DTW-mapped syllable intervals")
ax.set_xlabel("Frame")
ax.set_ylabel("Relative loudness")
ax.legend()
plt.tight_layout()


In [ ]:
# Structured output that can be consumed by downstream components
print(json.dumps(diagnostic, indent=2))

if diagnostic["signal_quality"]["low_quality"]:
    print("\nFeedback policy: LOW QUALITY SIGNAL -> avoid harsh feedback, request cleaner recording.")
elif diagnostic["stress_match"]:
    print("\nResult: Primary stress matches expected syllable.")
else:
    print("\nResult: Primary stress mismatch detected.")
